# Fase 3 : Módulo Supervisado Competitivo

- Clasificación: Árbol de Decisión vs Random Forest → `estado_desaparecido` (F1-Score)
- Regresión: Árbol de Decisión vs Random Forest → `dias_solucion` (MSE, R²)

: Persistencia de modelos y métricas en MongoDB.

In [11]:
# ============================================================================
# BLOQUE 1: EXTRACCIÓN DE DATOS PARA MODELADO SUPERVISADO
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_absolute_error, r2_score

# Conexión a la base de datos
engine = create_engine('mssql+pyodbc://DESKTOP-Q2AD3LV/DB_PersonasDesaparecidas?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes')

# CORRECCIÓN: Se mapea 'p.sexo AS genero' para adaptarlo al esquema de tu Dim_Persona
query = """
SELECT 
    p.edad,
    p.sexo AS genero,
    g.provincia,
    f.dias_solucion,
    f.situacion_actual
FROM Fact_Casos f
INNER JOIN Dim_Persona p ON f.id_persona = p.id_persona
INNER JOIN Dim_Geografia g ON f.id_geografia = g.id_geografia;
"""

with engine.connect() as connection:
    df_cases = pd.read_sql(query, con=connection)

print(f">>> Dataset extraído para entrenamiento. Total de registros: {df_cases.shape[0]}")

>>> Dataset extraído para entrenamiento. Total de registros: 75680


In [12]:
# ============================================================================
# BLOQUE 2: PREPROCESAMIENTO Y PIPELINE DE DATOS
# ============================================================================
# 1. Definir el target para el Clasificador (1 si ya fue encontrado/fallecido, 0 si sigue desaparecido)
df_cases['target_localizado'] = df_cases['situacion_actual'].apply(lambda x: 0 if x in ['DESAPARECIDO', 'EN INVESTIGACION'] else 1)

# 2. Separar características (Features) y Target
X = df_cases[['edad', 'genero', 'provincia']]
y_class = df_cases['target_localizado']

# 3. Definir transformadores para datos numéricos y categóricos
numeric_features = ['edad']
categorical_features = ['genero', 'provincia']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# 4. Dividir en set de Entrenamiento y Prueba
X_train, X_test, y_train_class, y_test_class = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)

# Aplicar las transformaciones matemáticas
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(">>> Preprocesamiento completado exitosamente.")
print(f"Dimensiones de entrenamiento: {X_train_processed.shape}")

>>> Preprocesamiento completado exitosamente.
Dimensiones de entrenamiento: (60544, 27)


### Fase 1 - Entrenamiento del Clasificador (¿Será localizado?)

In [13]:
# ============================================================================
#  CLASIFICACIÓN COMPETITIVA CORREGIDA (CONTROL DE DESBALANCE)
# ============================================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
import pymongo
from datetime import datetime

print(">>> Iniciando Clasificación Competitiva con Balanceo de Clases Histórico...")

# 1. Inicializar modelos inyectando penalización por desbalance de clases (class_weight)
dt_clf = DecisionTreeClassifier(max_depth=10, random_state=42, class_weight='balanced')
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')

# 2. Entrenar modelos
dt_clf.fit(X_train_processed, y_train_class)
rf_clf.fit(X_train_processed, y_train_class)

# 3. Predecir
y_pred_dt = dt_clf.predict(X_test_processed)
y_pred_rf = rf_clf.predict(X_test_processed)

# Evaluar usando F1-Score Macro para penalizar si una clase queda desamparada
f1_dt = f1_score(y_test_class, y_pred_dt, average='macro')
f1_rf = f1_score(y_test_class, y_pred_rf, average='macro')

print("\n======= REPORTE DE COMPETENCIA (F1-SCORE MACRO) =======")
print(f" ➔ F1-Score Árbol de Decisión: {f1_dt:.4f}")
print(f" ➔ F1-Score Random Forest: {f1_rf:.4f}")

print("\n======= DETALLE DE RENDIMIENTO - RANDOM FOREST =======")
print(classification_report(y_test_class, y_pred_rf))

ganador_clf = "Random Forest" if f1_rf > f1_dt else "Decision Tree"

# 4. Persistencia Automatizada en MongoDB
client = pymongo.MongoClient("mongodb://localhost:27017/")
db_mongo = client["DB_PersonasDesaparecidas_NoSQL"]
col_metricas = db_mongo["Metricas_Modelos"]

documento_clasificacion = {
    "experimento": "Clasificacion_Competitiva_Balanced_V2",
    "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "target": "target_localizado",
    "modelos_evaluados": {
        "DecisionTree": {"f1_score_macro": float(f1_dt)},
        "RandomForest": {"f1_score_macro": float(f1_rf)}
    },
    "algoritmo_ganador": ganador_clf,
    "tipo_modelo": "Clasificacion"
}

col_metricas.update_one({"experimento": "Clasificacion_Competitiva_Balanced_V2"}, {"$set": documento_clasificacion}, upsert=True)
print(f"\n>>> ¡Paso 3.4 completado! Métricas guardadas en MongoDB. Ganador: {ganador_clf}")

>>> Iniciando Clasificación Competitiva con Balanceo de Clases Histórico...

======= REPORTE DE COMPETENCIA (F1-SCORE MACRO) =======
 ➔ F1-Score Árbol de Decisión: 0.4900
 ➔ F1-Score Random Forest: 0.4969

======= DETALLE DE RENDIMIENTO - RANDOM FOREST =======
              precision    recall  f1-score   support

           0       0.08      0.66      0.15       513
           1       0.98      0.74      0.85     14623

    accuracy                           0.74     15136
   macro avg       0.53      0.70      0.50     15136
weighted avg       0.95      0.74      0.82     15136


>>> ¡Paso 3.4 completado! Métricas guardadas en MongoDB. Ganador: Random Forest


### Fase 2 - Regresión Acotada (¿En cuántos días?)

In [14]:
# ============================================================================
#  REGRESIÓN COMPETITIVA ADAPTADA (TIEMPOS DE RESPUESTA DE DENUNCIA)
# ============================================================================
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pymongo
from datetime import datetime

print(">>> Iniciando Fase 3.3: Regresión Competitiva (Métrica: Tiempo de Reporte)...")

# 1. Extraer directamente las fechas desde SQL para calcular la brecha de denuncia en caliente
query_fechas = """
SELECT 
    p.edad,
    p.sexo AS genero,
    g.provincia,
    f.fecha_desaparicion,
    f.fecha_denuncia
FROM Fact_Casos f
INNER JOIN Dim_Persona p ON f.id_persona = p.id_persona
INNER JOIN Dim_Geografia g ON f.id_geografia = g.id_geografia;
"""

with engine.connect() as connection:
    df_fechas = pd.read_sql(query_fechas, con=connection)

# 2. Calcular la diferencia real en días para la denuncia
df_fechas['fecha_desaparicion'] = pd.to_datetime(df_fechas['fecha_desaparicion'], errors='coerce')
df_fechas['fecha_denuncia'] = pd.to_datetime(df_fechas['fecha_denuncia'], errors='coerce')
df_fechas['dias_denuncia'] = (df_fechas['fecha_denuncia'] - df_fechas['fecha_desaparicion']).dt.days

# Filtro de calidad: Quedarse con registros lógicos mayores a 0 días de retraso
df_reg_final = df_fechas[(df_fechas['dias_denuncia'] >= 0) & (df_fechas['dias_denuncia'] <= 365)].copy()

print(f" ➔ Registros válidos encontrados para análisis de regresión: {df_reg_final.shape[0]}")

# 3. Dividir Variables y Procesar con el Pipeline Extendido
X_r = df_reg_final[['edad', 'genero', 'provincia']]
y_r = df_reg_final['dias_denuncia']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

X_train_r_proc = preprocessor.fit_transform(X_train_r)
X_test_r_proc = preprocessor.transform(X_test_r)

# 4. Inicializar Modelos Competitivos de Regresión
dt_reg = DecisionTreeRegressor(max_depth=10, random_state=42)
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)

# 5. Entrenar
dt_reg.fit(X_train_r_proc, y_train_r)
rf_reg.fit(X_train_r_proc, y_train_r)

# 6. Predecir y Evaluar Métricas Formales
y_pred_dt_r = dt_reg.predict(X_test_r_proc)
y_pred_rf_r = rf_reg.predict(X_test_r_proc)

mse_dt = mean_squared_error(y_test_r, y_pred_dt_r)
r2_dt = r2_score(y_test_r, y_pred_dt_r)

mse_rf = mean_squared_error(y_test_r, y_pred_rf_r)
r2_rf = r2_score(y_test_r, y_pred_rf_r)

print("\n======= REPORTES DE REGRESIÓN COMPETITIVA =======")
print(f" ➔ Árbol de Decisión -> MSE: {mse_dt:.4f} | R²: {r2_dt:.4f}")
print(f" ➔ Random Forest     -> MSE: {mse_rf:.4f} | R²: {r2_rf:.4f}")

ganador_reg = "Random Forest" if r2_rf > r2_dt else "Decision Tree"

# 7. Persistencia de Métricas de Regresión en MongoDB (Paso 3.4)
client = pymongo.MongoClient("mongodb://localhost:27017/")
db_mongo = client["DB_PersonasDesaparecidas_NoSQL"]
col_metricas = db_mongo["Metricas_Modelos"]

documento_regresion = {
    "experimento": "Regresion_Competitiva_Denuncias_V2",
    "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "target": "dias_denuncia",
    "modelos_evaluados": {
        "DecisionTree": {"mse": float(mse_dt), "r2": float(r2_dt)},
        "RandomForest": {"mse": float(mse_rf), "r2": float(r2_rf)}
    },
    "algoritmo_ganador": ganador_reg,
    "tipo_modelo": "Regresion"
}

col_metricas.update_one({"experimento": "Regresion_Competitiva_Denuncias_V2"}, {"$set": documento_regresion}, upsert=True)
print(f"\n>>> ¡Paso 3.4 completado! Métricas guardadas en MongoDB. Ganador de Regresión: {ganador_reg}")

>>> Iniciando Fase 3.3: Regresión Competitiva (Métrica: Tiempo de Reporte)...
 ➔ Registros válidos encontrados para análisis de regresión: 75370

======= REPORTES DE REGRESIÓN COMPETITIVA =======
 ➔ Árbol de Decisión -> MSE: 573.1905 | R²: -0.0245
 ➔ Random Forest     -> MSE: 560.9338 | R²: -0.0026

>>> ¡Paso 3.4 completado! Métricas guardadas en MongoDB. Ganador de Regresión: Random Forest


In [15]:
# ============================================================================
# BLOQUE DE EXPORTACIÓN: PERSISTENCIA DE MODELOS SUPERVISADOS EN LA RAÍZ
# ============================================================================
import joblib
import os

# Retroceder dos niveles de forma segura: de 03_supervisado -> notebooks -> raíz del proyecto
base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
models_dir = os.path.join(base_dir, "models")

# Asegurar que la carpeta 'models' principal exista
os.makedirs(models_dir, exist_ok=True)

# Definir las rutas exactas de guardado (Cambia 'rf_clf', 'rf_reg' y 'preprocessor' por los nombres de tus variables si mapeaste otros)
path_classifier = os.path.join(models_dir, "rf_classifier.pkl")
path_regressor = os.path.join(models_dir, "rf_regressor.pkl")
path_preprocessor = os.path.join(models_dir, "supervisado_preprocessor.pkl")

# Exportar los archivos binarios (.pkl)
joblib.dump(rf_clf, path_classifier)
joblib.dump(rf_reg, path_regressor)
joblib.dump(preprocessor, path_preprocessor)

print(f">>> ¡Modelos Supervisados y Preprocesador exportados con éxito en: {models_dir}!")

>>> ¡Modelos Supervisados y Preprocesador exportados con éxito en: c:\Users\HP\OneDrive\proyecto-personas-desaparecidas\models!
